# epimux tutorial

A complete bulk multi-omic differential analysis, from count matrices to a
report — and, more importantly, the checks that decide whether the result can be
believed.

We use synthetic data with a **known ground truth**, so every claim can be
verified against what we planted.


## 1. Build a dataset

Everything is anchored on one reference element set. Each assay keeps its own
native features and is projected onto that reference, so layers become directly
comparable without repeated interval arithmetic.

The simulation has three groups:

* `WT` and `KO` — the contrast of interest; 300 of 4,000 elements truly change.
* `OTHER` — a different cell type, where a **large** fraction of elements differ.
  This is what a positive control needs: a comparison you already know differs.


In [ ]:
import numpy as np
import pandas as pd
import epimux as ep

rng = np.random.default_rng(0)
N, N_CHANGE = 4000, 300

elements = pd.DataFrame({
    "chrom": "chr1",
    "start": np.arange(N) * 5_000,
    "end":   np.arange(N) * 5_000 + 1_000,
})


def make_counts(effect, seed):
    """Three groups over the same elements.

    KO gains `effect`-fold at the first N_CHANGE elements. OTHER is a different
    cell type: a broad shift drawn ONCE for the group, so its replicates agree
    with each other -- which is what makes it usable as a positive control.
    """
    r = np.random.default_rng(seed)
    base = r.lognormal(4.4, 0.9, N)
    cell_type_shift = r.lognormal(0, 0.8, N)        # drawn once, not per replicate

    cols, design = {}, {"WT": [], "KO": [], "OTHER": []}
    for g in ("WT", "KO", "OTHER"):
        mu = base.copy()
        if g == "KO":
            mu[:N_CHANGE] *= effect                 # <-- the ground truth
        elif g == "OTHER":
            mu = mu * cell_type_shift
        for i in (1, 2, 3):
            cols[f"{g}_R{i}"] = r.poisson(mu)
            design[g].append(f"{g}_R{i}")
    return pd.DataFrame(cols, index=[f"e{i}" for i in range(N)]), design


atac, design = make_counts(2.2, seed=1)
h3k,  _      = make_counts(2.0, seed=2)

ds = ep.Dataset(elements, genome="synthetic", name="tutorial")
ds.add_counts("ATAC",    intervals=elements.set_index(atac.index), matrix=atac)
ds.add_counts("H3K27ac", intervals=elements.set_index(h3k.index),  matrix=h3k)
ds.set_design(design)
ds.summary()


## 2. Differential analysis

`log2FC` is **always** `log2(test / ref)`. The direction is carried by a
`Contrast` object rather than inferred from factor levels — and it is re-verified
against the raw counts before the result is returned.

This matters more than it sounds. In R, `factor(c("WT","KO"))` sorts its levels
alphabetically to `("KO","WT")`, so a contrast written as "WT vs KO" silently
computes `log2(WT/KO)`. Every reported direction inverts, and nothing downstream
complains: the p-values are identical and the volcano plot looks normal.


In [ ]:
res = ds.differential(ref="WT", test="KO")
res["ATAC"].head()


In [ ]:
# the planted elements (0-299) should be the significant, UP ones
sig = ds.significant("ATAC")
planted = int((sig.index < N_CHANGE).sum())
up = int((sig["log2FC"] > 0).sum())

print(f"significant elements: {len(sig)}")
print(f"  planted (index < {N_CHANGE}): {planted}   "
      f"({100 * planted / len(sig):.0f}% of calls are true positives)")
print(f"  in the UP direction:     {up} / {len(sig)}")
print(f"  median log2FC: {sig['log2FC'].median():+.2f}"
      f"  (we planted 2.2x = {np.log2(2.2):+.2f})")
print(f"recovered {planted}/{N_CHANGE} of the planted set")


## 3. The audit

`audit()` is what separates a trustworthy result from a plausible one. Each check
corresponds to a way real analyses go wrong.

Note the positive control: it is `("WT", "OTHER")` — a **different** comparison,
one we know differs. Using the contrast of interest as its own positive control
would be circular and tells you nothing.


In [ ]:
ds.audit(positive_control=("WT", "OTHER"), null_group="WT")
ds.audit_report.to_frame()


Read the table as a whole:

* `check_direction` — the reported sign agrees with the raw counts.
* `positive_control` — the pipeline detects a large fraction of the cell-type
  differences, so it demonstrably has power. **A pipeline that cannot find a
  difference you know exists cannot support a null result.**
* `null_contrast` — splitting `WT` replicates against each other yields almost
  nothing, so the real hits are not noise.
* `outlier_replicates` / `pvalue_diagnostic` — no single library dominates, and
  the p-value histogram has the healthy shape (spike at zero over a flat null).


## 4. Cross-layer coupling

The headline number in a multi-omic paper is usually a cross-layer correlation.
It is also easy to get wrong: correlate two results built from *opposite*
contrast orientations and a coordinated change reads as "decoupling".

`couple()` raises in that situation rather than returning a flipped number.


In [ ]:
c = ds.coupling("ATAC", "H3K27ac")
print(c)


In [ ]:
states = ds.classify()
ep.concordance(states)


States are called from **significance in each layer**, never from the sign of a
single noisy difference. Sign-only state calls are the classic source of
irreproducible "discordant element" lists — shuffle the replicates and the
membership changes.

Here both assays were given the same planted elements, so the expected answer is
`coordinated_up` for roughly the planted set and nothing discordant.


## 5. Normalization when a global shift is possible

Size-factor methods assume most features do not change. When that assumption
breaks — a genome-wide gain or loss — they absorb the effect and hand back a
confident null.


In [ ]:
ep.assess_global_shift(atac, ds.contrast("WT", "KO"))


In [ ]:
# with spike-ins the global magnitude survives normalization
spike = {"WT_R1": 1000, "WT_R2": 1100, "WT_R3": 950,
         "KO_R1": 1020, "KO_R2": 980,  "KO_R3": 1050}
ep.spike_in_factors(spike)


## 6. Power

How large an effect could this design detect, and how many replicates would the
observed effect have needed?


In [ ]:
ds.power("ATAC")


## 7. Cross-contrast comparison

"Is the effect the same in two settings?" is **not** answered by comparing two
significance lists — lists differ because of replicate count, depth and
dispersion. `compare_contrasts` tests the interaction per element.


In [ ]:
res_h3k = ds.results["H3K27ac"]
cmp = ep.compare_contrasts(ds.results["ATAC"], res_h3k, "ATAC", "H3K27ac")
summary = ep.concordance_summary(cmp, "ATAC", "H3K27ac")
{k: summary[k] for k in ("shared", "shared_same_direction",
                         "elements_with_interaction", "interpretation")}


## 8. Export and report


In [ ]:
ds.report("tutorial_report.html")


In [ ]:
import json
written = ds.export("tutorial_out")
json.load(open(written["manifest"]))["results"]


The manifest records the contrast, the sign convention, the audit outcome and
the per-assay summary — so the analysis can be checked months later without
re-running it.

## Where to go next

* [`guide-audit.md`](guide-audit.md) — what each check tests and how to act on it.
* [`guide-normalization.md`](guide-normalization.md) — spike-ins, and when a
  magnitude is simply unrecoverable.
* [`guide-hic.md`](guide-hic.md) — compartments, P(s), insulation, APA and
  contact-based gene linking. Hi-C needs no spike-in, so it is often the right
  way to ask about the *functional consequence* of a binding change when the
  ChIP itself is too shallow to quantify.
* [`guide-inputs.md`](guide-inputs.md) — every input format in detail.
